In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, roc_auc_score)
import pickle
import os

# Load processed data
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Loaded successfully")

# Set MLflow experiment
mlflow.set_experiment("churn_prediction")
print("MLflow experiment set")

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X_train: (5634, 44)
X_test: (1409, 44)
Loaded successfully


2026/06/10 13:00:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/10 13:00:15 INFO mlflow.store.db.utils: Updating database tables
2026/06/10 13:00:16 INFO mlflow.tracking.fluent: Experiment with name 'churn_prediction' does not exist. Creating a new experiment.


MLflow experiment set


In [2]:
# Function to evaluate model
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }

# Define 3 models to train
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':            XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

# Train and log each model
for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        # Train
        model.fit(X_train, y_train)
        
        # Evaluate
        metrics = evaluate_model(model, X_test, y_test)
        
        # Log params
        mlflow.log_param("model_type", model_name)
        
        # Log metrics
        for metric_name, value in metrics.items():
            mlflow.log_metric(metric_name, value)
        
        # Log model
        mlflow.sklearn.log_model(model, model_name)
        
        print(f"\n{model_name}")
        print(f"  F1:      {metrics['f1']:.4f}")
        print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
        print(f"  Recall:  {metrics['recall']:.4f}")

print("\nAll models trained and logged to MLflow")

2026/06/10 13:02:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 13:02:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



LogisticRegression
  F1:      0.5957
  ROC-AUC: 0.8407
  Recall:  0.5535


2026/06/10 13:02:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 13:02:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/10 13:02:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



RandomForest
  F1:      0.5277
  ROC-AUC: 0.8155
  Recall:  0.4706


2026/06/10 13:02:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



XGBoost
  F1:      0.5894
  ROC-AUC: 0.8195
  Recall:  0.5642

All models trained and logged to MLflow
